In [9]:
import pandas as pd
import zipfile
import io
import sys, os
from pathlib import Path
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
from tqdm import tqdm
project_root = "/home/shuol/ls_learnqf"
if project_root not in sys.path:
    sys.path.append(project_root)
from util import *
from stock_pool import *
import tushare as ts
import time
import pickle

经验证据表明，低波动股票收益表现往往优于高波动股票，股票收益与波动之间存在着负相关性。为了检验A股市场的低波动异象，选取振幅因子作为波动类因子的代理变量。\
为了进一步分析振幅因子的信息结构，引入价格维度。预期在不同价格位置，振幅分布所蕴含的信息会存在结构性差异。

回测区间为2010年4月30日至2020年4月30日；样本空间为全体A股，剔除ST股和上市未满60 日的新股；每月月初调仓，持仓一个自然月，交易费率千分之三。

In [7]:
# 初始化pro接口
pro = ts.pro_api('0afffc3e8d0c5d2d2478e749ee179c10c3efddac434f4dd953f57730')

# 生成历史样本空间
sample_space = get_history_sample_space_df(pro=pro, start_date='20100430', end_date='20200430', pool_type='ALL', min_list_days=60)

In [8]:
sample_space

,trade_date,ts_code
0,20160831,300373.SZ
1,20160831,600626.SH
2,20160831,601636.SH
3,20160831,600595.SH
4,20160831,600299.SH
...,...,...
143928,20200430,600928.SH
143929,20200430,603059.SH
143930,20200430,603109.SH
143931,20200430,002916.SZ


**振幅因子** ：回看最近20个交易日，计算股票每日的振幅（最高价/最低价-1），取其均值作为振幅因子。\
\
价格维度下的振幅因子切割方案：
1. 数据回溯  
   对选定股票 $ S $，回溯获取其最近 $N$ 个交易日的数据（研报里 $N = 20$）。
2. 振幅计算
   计算股票 $ S $每日的振幅：
   $
   \text{振幅} = \frac{\text{最高价}}{\text{最低价}} - 1
   $
3. 高价振幅因子 $V_{\text{high}}(\lambda)$
   选择收盘价较高的前 $\lambda $（例如 40%）的有效交易日，计算这些交易日振幅的均值，得到高价振幅因子：
   $
   V_{\text{high}}(\lambda)
   $
4. 低价振幅因子 $ V_{\text{low}}(\lambda) $
   选择收盘价较低的前 $\lambda $（例如 40%）的有效交易日，计算这些交易日振幅的均值，得到低价振幅因子：
   $
   V_{\text{low}}(\lambda)
   $

In [ ]:
def calculate_amplitude_factors(daily_data: pd.DataFrame, N: int = 20, lambda_ratio: float = 0.4) -> pd.DataFrame:
    """
    计算价格维度下的振幅因子
    :param daily_data: 包含过去 N 天日线数据的 DataFrame，必须有 ['ts_code', 'trade_date', 'high', 'low', 'close']
    :param N: 回溯天数 (默认 20)
    :param lambda_ratio: 高低价区域占比 (默认 0.4)
    :return: 包含 V_all, V_high, V_low 的因子 DataFrame
    """
    # 1. 计算每日的基础振幅
    df = daily_data.copy()
    df['amp'] = df['high'] / df['low'] - 1.0
    
    # 2. 定义内部计算逻辑 (针对单只股票的 N 天数据)
    def calc_group_factors(group):
        # 有效天数
        n = len(group)
        if n < int(N * 0.5): 
            # 如果这只股票过去20天里停牌太多，有效数据不足一半，则返回空值
            return pd.Series({'V_all': np.nan, 'V_high': np.nan, 'V_low': np.nan})
            
        # 计算需要截取的天数 k (例如 20 * 0.4 = 8天)
        k = int(np.round(n * lambda_ratio))
        
        # 整体振幅均值 (回溯期内的平均振幅)
        v_all = group['amp'].mean()
        
        # 按收盘价从小到大排序，分离高低价状态
        sorted_group = group.sort_values('close')
        
        # 低价区振幅 V_low: 收盘价最低的 k 天的振幅均值
        v_low = sorted_group['amp'].iloc[:k].mean() if k > 0 else np.nan
        
        # 高价区振幅 V_high: 收盘价最高的 k 天的振幅均值
        v_high = sorted_group['amp'].iloc[-k:].mean() if k > 0 else np.nan
        
        return pd.Series({'V_all': v_all, 'V_high': v_high, 'V_low': v_low})

    # 3. 按股票代码分组计算因子
    factor_df = df.groupby('ts_code').apply(calc_group_factors).reset_index()
    
    return factor_df

高价振幅因子具有更强的负向选股能力, 但年化波动率和最大回撤也相对较高。\
为了提升高价振幅因子$V_{\text{high}}$ 的选股稳定性，考虑在横截面上对高价振幅因子进行标准化处理。\
这里标准化的做法是：在同一切割比例$\lambda$下，我们将高价振幅
因子$V_{\text{high}}$ 与低价振幅因子$V_{\text{low}}$ 作差，构造得到**理想振幅因子$V$**，表达式如下：
$$V(\lambda) = V_{\text{high}}(\lambda) - V_{\text{low}}(\lambda)$$

理想振幅因子的选股能力要显著优于高价振幅因子。现在进一步因子测评。\
对理想振幅因子进行行业风格中性化：剔除行业和主要风格因子（市值、动量、波动率、流动性、Beta）。

考察理想振幅因子对参数回看天数$N$的敏感性

不同样本空间：沪深300、中证500、中证1000。

振幅和换手率都是反映股票成交活跃程度的指标。换手率因子是否也具有同样的隐藏结构？\
使用过去20日换手率均值代表换手率因子，应用理想振幅因子的构造框架构造理想换手率因子。